#  COVID-19 ClaimCheck — Structured EDA Notebook

**Workflow:**
```
Load Data → Understand Data → Clean Data → Statistical Analysis → Visualization → Find Patterns & Relationships → Prepare for Modeling
```

**Dataset:** COVID-19 Open Research Dataset (CORD-19) — research abstracts for claim verification


---
## ⚙️ Step 0 — Install & Import Libraries

In [ ]:
# !pip install pandas numpy matplotlib seaborn wordcloud nltk scikit-learn

import os, re, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter

from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)

# ── Global plot style ──────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
COLORS = sns.color_palette("Set2", 10)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "figure.autolayout": True})

print("All libraries loaded successfully.")

All libraries loaded successfully.


---
## STEP 1 — Load Data

> **Goal:** Load the COVID-19 CORD-19 dataset into a DataFrame.  
> The demo generator produces realistic synthetic data so every cell runs immediately.
> To use the real dataset, download `metadata.csv` from https://www.semanticscholar.org/cord19/download


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_cord19_metadata(csv_path: str) -> pd.DataFrame:
    """Load real CORD-19 metadata.csv and return records with abstracts."""
    df = pd.read_csv(csv_path, low_memory=False)
    df = df.dropna(subset=["abstract"])
    df = df[df["abstract"].str.strip() != ""]
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df):,} abstracts from CORD-19.")
    return df


def create_demo_dataframe(n: int = 3_000) -> pd.DataFrame:
    """Generate a realistic synthetic CORD-19-style dataframe for demo/testing."""
    rng = np.random.default_rng(42)

    topics = [
        "vaccine efficacy immunization",
        "mask wearing transmission reduction",
        "hydroxychloroquine treatment",
        "remdesivir antiviral therapy",
        "PCR testing diagnosis",
        "lockdown social distancing policy",
        "long COVID post-acute sequelae",
        "variant mutation spike protein",
        "ICU mortality severe illness",
        "aerosol airborne transmission",
        "herd immunity seroprevalence",
        "mRNA vaccine side effects",
    ]

    sentences = {
        "vaccine efficacy immunization": [
            "Vaccine efficacy against severe disease remained high across all age groups.",
            "Immunization programs were associated with a significant reduction in hospitalizations.",
            "Two-dose regimens demonstrated stronger immune responses than single-dose protocols.",
            "Booster doses restored waning immunity in elderly populations.",
            "Antibody titers were significantly elevated following vaccination.",
        ],
        "mask wearing transmission reduction": [
            "Consistent mask use was correlated with lower community transmission rates.",
            "N95 respirators provided superior filtration compared to cloth masks.",
            "Mask mandates in schools reduced infection rates among children.",
            "Surgical masks reduced droplet spread effectively in clinical settings.",
            "Community compliance with masking protocols varied significantly by region.",
        ],
        "hydroxychloroquine treatment": [
            "Randomized controlled trials found no significant benefit of hydroxychloroquine.",
            "Early observational studies reported mixed findings on chloroquine efficacy.",
            "Drug repurposing efforts targeted existing antivirals for COVID-19 treatment.",
            "Adverse cardiac events were noted in some hydroxychloroquine recipients.",
            "WHO halted the hydroxychloroquine arm of the Solidarity trial.",
        ],
        "remdesivir antiviral therapy": [
            "Remdesivir shortened the duration of hospitalization in moderate COVID-19 cases.",
            "The antiviral drug showed in vitro activity against SARS-CoV-2.",
            "Clinical trials assessed remdesivir in combination with corticosteroids.",
            "Intravenous remdesivir was approved for emergency use in hospitalized patients.",
            "Liver enzyme elevations were observed in a subset of treated patients.",
        ],
        "PCR testing diagnosis": [
            "RT-PCR remains the gold standard for SARS-CoV-2 diagnosis.",
            "Rapid antigen tests demonstrated high specificity but lower sensitivity.",
            "Cycle threshold values were explored as surrogates for viral load.",
            "Saliva-based PCR showed comparable accuracy to nasopharyngeal swabs.",
            "Testing capacity constraints limited early pandemic surveillance efforts.",
        ],
        "lockdown social distancing policy": [
            "Strict lockdown measures were associated with flattening the epidemic curve.",
            "School closures reduced pediatric transmission but had educational costs.",
            "Physical distancing of at least one meter reduced transmission risk.",
            "Economic disruptions were disproportionate among low-income populations.",
            "Policy adherence declined over time during prolonged restriction periods.",
        ],
        "long COVID post-acute sequelae": [
            "Persistent fatigue was the most commonly reported long COVID symptom.",
            "Cognitive impairment, termed 'brain fog', affected many recovered patients.",
            "Post-acute sequelae were more prevalent in patients with severe acute illness.",
            "Pulmonary function abnormalities persisted for months after discharge.",
            "Long COVID disproportionately affected working-age adults.",
        ],
        "variant mutation spike protein": [
            "The Delta variant carried mutations that enhanced ACE2 receptor binding.",
            "Spike protein mutations were associated with increased transmissibility.",
            "Omicron subvariants showed significant immune evasion properties.",
            "Whole genome sequencing was essential for tracking variant emergence.",
            "Cross-reactive T-cell responses were preserved against most variants.",
        ],
        "ICU mortality severe illness": [
            "ICU mortality rates were highest among older adults with comorbidities.",
            "Mechanical ventilation was required in a significant proportion of ICU admissions.",
            "Prone positioning improved oxygenation in mechanically ventilated patients.",
            "Dexamethasone reduced 28-day mortality in patients requiring respiratory support.",
            "Healthcare capacity strain increased case fatality rates during peak surges.",
        ],
        "aerosol airborne transmission": [
            "Aerosol-generating procedures significantly increased nosocomial infection risk.",
            "Indoor ventilation was identified as a key factor in airborne transmission.",
            "Fine aerosols could remain suspended in poorly ventilated spaces for hours.",
            "HEPA filtration reduced airborne viral load in clinical environments.",
            "Superspreading events were linked to enclosed, crowded settings.",
        ],
        "herd immunity seroprevalence": [
            "Seroprevalence studies estimated the true extent of community infection.",
            "Herd immunity thresholds were revised upward with the emergence of variants.",
            "Antibody waning reduced the duration of natural immunity post-infection.",
            "Hybrid immunity from infection plus vaccination offered the strongest protection.",
            "Population-level seroprevalence varied substantially by geographic region.",
        ],
        "mRNA vaccine side effects": [
            "Reactogenicity was more common after the second mRNA vaccine dose.",
            "Rare cases of myocarditis were reported following mRNA vaccination in adolescents.",
            "Injection-site pain was the most frequently reported local side effect.",
            "Systemic reactions such as fever resolved within 48 hours in most recipients.",
            "Anaphylaxis rates were extremely low, estimated at fewer than 5 per million doses.",
        ],
    }

    topic_choices = rng.choice(topics, size=n)
    years = rng.choice([2020, 2021, 2022, 2023], size=n, p=[0.25, 0.35, 0.25, 0.15])
    journals = rng.choice(
        ["Nature Medicine", "Lancet", "NEJM", "JAMA", "BMJ",
         "Science", "Cell", "PLOS ONE", "medRxiv", "bioRxiv"], size=n
    )
    claim_labels = rng.choice(
        ["Supported", "Refuted", "Not Enough Information"],
        size=n, p=[0.45, 0.25, 0.30]
    )

    rows = []
    for i, topic in enumerate(topic_choices):
        base = sentences[topic]
        sents = list(rng.choice(base, size=rng.integers(3, 7), replace=True))
        sents.append(f"This study presents findings relevant to {topic.split()[0]} research.")
        rows.append({
            "paper_id"      : f"CORD-{i:05d}",
            "title"         : f"Study on {topic.title()} ({years[i]})",
            "abstract"      : " ".join(sents),
            "authors"       : f"Author{rng.integers(1,50)}, Author{rng.integers(51,100)}",
            "publish_year"  : int(years[i]),
            "journal"       : journals[i],
            "topic"         : topic,
            "claim_label"   : claim_labels[i],
        })

    df = pd.DataFrame(rows)
    print(f"Demo dataframe created: {len(df):,} records.")
    return df


# ── CHOOSE ONE ──────────────────────────────────────────────
USE_REAL_DATA = True   # ← Set True and provide CORD19_PATH for real data
CORD19_PATH = "D:/4th year/8th sem/NLP/Project/datasets/metadata.csv"  # ← Change this to your file's path


if USE_REAL_DATA:
    df = load_cord19_metadata(CORD19_PATH)
else:
    df = create_demo_dataframe(n=3_000)

print(f"\nDataset shape: {df.shape}")
df.head(5)

Loaded 821,116 abstracts from CORD-19.

Dataset shape: (821116, 19)


,cord_uid,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,mag_id,who_covidence_id,arxiv_id,pdf_json_files,pmc_json_files,url,s2_id
0,ug7v899j,d1aafb70c066a2068b02786f8929fd9c900897fb,PMC,Clinical features of culture-proven Mycoplasma...,10.1186/1471-2334-1-6,PMC35282,11472636,no-cc,OBJECTIVE: This retrospective chart review des...,2001-07-04,"Madani, Tariq A; Al-Ghamdi, Aisha A",BMC Infect Dis,NaN,NaN,NaN,document_parses/pdf_json/d1aafb70c066a2068b027...,document_parses/pmc_json/PMC35282.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3...,NaN
1,02tnwd4m,6b0567729c2143a66d737eb0a2f63f2dce2e5a7d,PMC,Nitric oxide: a pro-inflammatory mediator in l...,10.1186/rr14,PMC59543,11667967,no-cc,Inflammatory diseases of the respiratory tract...,2000-08-15,"Vliet, Albert van der; Eiserich, Jason P; Cros...",Respir Res,NaN,NaN,NaN,document_parses/pdf_json/6b0567729c2143a66d737...,document_parses/pmc_json/PMC59543.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,NaN
2,ejv2xln0,06ced00a5fc04215949aa72528f2eeaae1d58927,PMC,Surfactant protein-D and pulmonary host defense,10.1186/rr19,PMC59549,11667972,no-cc,Surfactant protein-D (SP-D) participates in th...,2000-08-25,"Crouch, Erika C",Respir Res,NaN,NaN,NaN,document_parses/pdf_json/06ced00a5fc04215949aa...,document_parses/pmc_json/PMC59549.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,NaN
3,2b73a28n,348055649b6b8cf2b9a376498df9bf41f7123605,PMC,Role of endothelin-1 in lung disease,10.1186/rr44,PMC59574,11686871,no-cc,Endothelin-1 (ET-1) is a 21 amino acid peptide...,2001-02-22,"Fagan, Karen A; McMurtry, Ivan F; Rodman, David M",Respir Res,NaN,NaN,NaN,document_parses/pdf_json/348055649b6b8cf2b9a37...,document_parses/pmc_json/PMC59574.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,NaN
4,9785vg6d,5f48792a5fa08bed9f56016f4981ae2ca6031b32,PMC,Gene expression in epithelial cells in respons...,10.1186/rr61,PMC59580,11686888,no-cc,Respiratory syncytial virus (RSV) and pneumoni...,2001-05-11,"Domachowske, Joseph B; Bonville, Cynthia A; Ro...",Respir Res,NaN,NaN,NaN,document_parses/pdf_json/5f48792a5fa08bed9f560...,document_parses/pmc_json/PMC59580.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,NaN


---
## 🔍 STEP 2 — Understand Data

> **Goal:** Get a complete picture of shape, columns, types, missing values, and sample records before touching any data.

In [6]:
# 2.1  Basic shape & column info
print("━" * 55)
print("  BASIC OVERVIEW")
print("━" * 55)
print(f"  Rows      : {df.shape[0]:,}")
print(f"  Columns   : {df.shape[1]}")
print(f"  Columns   : {list(df.columns)}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BASIC OVERVIEW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Rows      : 821,116
  Columns   : 19
  Columns   : ['cord_uid', 'sha', 'source_x', 'title', 'doi', 'pmcid', 'pubmed_id', 'license', 'abstract', 'publish_time', 'authors', 'journal', 'mag_id', 'who_covidence_id', 'arxiv_id', 'pdf_json_files', 'pmc_json_files', 'url', 's2_id']


In [7]:
# 2.2  Data types
print("\nData Types:")
print(df.dtypes)


Data Types:
cord_uid             object
sha                  object
source_x             object
title                object
doi                  object
pmcid                object
pubmed_id            object
license              object
abstract             object
publish_time         object
authors              object
journal              object
mag_id              float64
who_covidence_id     object
arxiv_id             object
pdf_json_files       object
pmc_json_files       object
url                  object
s2_id               float64
dtype: object


In [8]:
print(df.columns)

Index(['cord_uid', 'sha', 'source_x', 'title', 'doi', 'pmcid', 'pubmed_id',
       'license', 'abstract', 'publish_time', 'authors', 'journal', 'mag_id',
       'who_covidence_id', 'arxiv_id', 'pdf_json_files', 'pmc_json_files',
       'url', 's2_id'],
      dtype='object')


In [9]:
# 2.3  Missing values summary
# 2.3 Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})
pd.set_option('display.max_rows', None)

print("\nMissing Values:")
print(missing_df)


Missing Values:
                  Missing Count  Missing %
cord_uid                      0       0.00
sha                      494432      60.21
source_x                      0       0.00
title                       111       0.01
doi                      286423      34.88
pmcid                    508708      61.95
pubmed_id                416693      50.75
license                       0       0.00
abstract                      0       0.00
publish_time               1723       0.21
authors                    5023       0.61
journal                   78875       9.61
mag_id                   821116     100.00
who_covidence_id         462033      56.27
arxiv_id                 806867      98.26
pdf_json_files           494432      60.21
pmc_json_files           549771      66.95
url                      260542      31.73
s2_id                     55819       6.80


In [10]:
df = df.rename(columns={
    'sha': 'paper_id',
    'publish_time': 'published_date',
    'pubmed_id': 'pubmed',
    'pmcid': 'pmc_id',
    's2_id': 'semantic_scholar_id'
})

In [11]:
# 2.4  Duplicate check
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
print(f"Unique paper IDs: {df['paper_id'].nunique():,}")

Duplicate rows: 0
Unique paper IDs: 326,644


In [12]:
# 2.5  Categorical cardinality
cat_cols = df.select_dtypes(include="object").columns
print("\nCategorical Column Cardinality:")
for col in cat_cols:
    print(f"  {col:<20}: {df[col].nunique()} unique values")


Categorical Column Cardinality:
  cord_uid            : 764494 unique values
  paper_id            : 326644 unique values
  source_x            : 49 unique values
  title               : 677668 unique values
  doi                 : 533747 unique values
  pmc_id              : 312408 unique values
  pubmed              : 403977 unique values
  license             : 18 unique values
  abstract            : 730712 unique values
  published_date      : 7644 unique values
  authors             : 669256 unique values
  journal             : 50158 unique values
  who_covidence_id    : 359083 unique values
  arxiv_id            : 14249 unique values
  pdf_json_files      : 326644 unique values
  pmc_json_files      : 271345 unique values
  url                 : 560574 unique values


In [13]:
# 2.6  Sample records
print("\nSample Abstract:")
print(df["abstract"].iloc[0])
print()



Sample Abstract:
OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia. METHODS: Patients with positive M. pneumoniae cultures from respiratory specimens from January 1997 through December 1998 were identified through the Microbiology records. Charts of patients were reviewed. RESULTS: 40 patients were identified, 33 (82.5%) of whom required admission. Most infections (92.5%) were community-acquired. The infection affected all age groups but was most common in infants (32.5%) and pre-school children (22.5%). It occurred year-round but was most common in the fall (35%) and spring (30%). More than three-quarters of patients (77.5%) had comorbidities. Twenty-four isolates (60%) were associated with pneumonia, 14 (35%) with upper respiratory tract infections, and 2 (5%) with bronchiolitis. Cough (82.5%), fever (75%), and mala

In [14]:
df.describe(include="all")

,cord_uid,paper_id,source_x,title,doi,pmc_id,pubmed,license,abstract,published_date,authors,journal,mag_id,who_covidence_id,arxiv_id,pdf_json_files,pmc_json_files,url,semantic_scholar_id
count,821116,326684,821116,821005,534693,312408,404423,821116,821116,819393,816093,742241,0.0,359083,14249,326684,271345,560574,7.652970e+05
unique,764494,326644,49,677668,533747,312408,403977,18,730712,7644,669256,50158,NaN,359083,14249,326644,271345,560574,NaN
top,3xm3rbz5,31bc0fb718edaab9e33f678909710f62c40abebc,WHO,ACR Convergence 2020 Abstract Supplement,10.1016/j.scitotenv.2020.139397,PMC7427695,33117894,unk,[Figure: see text].,2021,"Anonymous,",PLoS One,NaN,#mdl-35386817,2106.02118,document_parses/pdf_json/31bc0fb718edaab9e33f6...,document_parses/pmc_json/PMC7427695.xml.json,https://doi.org/10.1038/s41372-020-00775-z; ht...,NaN
freq,99,3,334611,99,9,1,5,445310,206,183517,1472,9937,NaN,1,1,3,1,1,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.164150e+08
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.667331e+07
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.600000e+01
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.215071e+08
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.323656e+08
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.380847e+08


In [ ]:
from collections import Counter
import re

# Combine all abstracts into one text
text = " ".join(df["abstract"].dropna().astype(str))

# Convert to lowercase
text = text.lower()

# Extract words
words = re.findall(r'\b[a-zA-Z-]+\b', text)

# Remove very common stopwords
stopwords = {
    'the', 'and', 'of', 'to', 'in', 'a', 'for', 'is', 'on',
    'with', 'that', 'by', 'as', 'are', 'at', 'an', 'be',
    'this', 'from', 'or', 'we', 'was', 'were', 'it'
}

filtered_words = [word for word in words if word not in stopwords]

# Count word frequency
word_counts = Counter(filtered_words)

# Show top 30 common words
print(word_counts.most_common(30))

In [ ]:
# COVID-related keywords
covid_keywords = [
    "covid", "covid-19", "coronavirus",
    "sars-cov-2", "2019-ncov", "pandemic"
]

# Keep only rows whose abstract contains COVID-related terms
df = df[
    df["abstract"].str.lower().str.contains(
        "|".join(covid_keywords),
        na=False
    )
]

# Reset index
df = df.reset_index(drop=True)

print(f"Remaining COVID-related papers: {len(df):,}")

---
## STEP 3 — Clean Data

> **Goal:** Fix quality issues, remove noise, and engineer structured text features from raw abstracts.

In [ ]:
# 3.1  Drop duplicates & blanks
original_len = len(df)
df = df.drop_duplicates()
df = df[df["abstract"].str.strip() != ""]
df = df.reset_index(drop=True)
print(f"Rows before: {original_len:,}  |  After cleaning: {len(df):,}  |  Dropped: {original_len - len(df)}")

Rows before: 821,116  |  After cleaning: 821,116  |  Dropped: 0


In [ ]:
# 3.2  Text Preprocessing — Lower, remove URLs/digits/punctuation, tokenize, lemmatize
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# COVID-domain stopwords that carry little discriminative signal
COVID_STOP = {"covid", "covid19", "sars", "cov", "study", "patients",
              "results", "data", "analysis", "paper", "research", "findings"}

def preprocess_text(text: str) -> str:
    """Clean and normalise a single abstract string."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)    # remove URLs
    text = re.sub(r"\d+", "", text)                 # remove digits
    text = re.sub(r"[^\w\s]", " ", text)           # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and t not in COVID_STOP and len(t) > 2
    ]
    return " ".join(tokens)

df["clean_abstract"] = df["abstract"].apply(preprocess_text)
print(f"Preprocessing complete. Example:")
print(f"  ORIGINAL : {df['abstract'].iloc[0][:120]}...")
print(f"  CLEANED  : {df['clean_abstract'].iloc[0][:120]}...")

Preprocessing complete. Example:
  ORIGINAL : OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-...
  CLEANED  : objective retrospective chart review describes epidemiology clinical feature culture proven mycoplasma pneumoniae infect...


In [ ]:
# 3.3  Engineer structured features from text
df["word_count"]        = df["abstract"].apply(lambda x: len(x.split()))
df["char_count"]        = df["abstract"].apply(len)
df["sentence_count"]    = df["abstract"].apply(lambda x: len(sent_tokenize(x)))
df["unique_word_count"] = df["clean_abstract"].apply(lambda x: len(set(x.split())))
df["avg_word_length"]   = df["abstract"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
df["vocab_richness"]    = df["unique_word_count"] / (df["word_count"] + 1)

print("\n Feature engineering complete. New features:")
print(df[["word_count","char_count","sentence_count",
          "unique_word_count","avg_word_length","vocab_richness"]].head(3))


 Feature engineering complete. New features:
   word_count  char_count  sentence_count  unique_word_count  avg_word_length  \
0         262        1847              15                 89         6.053435   
1         142        1001               5                 67         6.056338   
2         219        1647               8                 98         6.525114   

   vocab_richness  
0        0.338403  
1        0.468531  
2        0.445455  


In [ ]:
# 3.4  Detect & flag outliers (IQR method on word_count)
Q1, Q3 = df["word_count"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_fence, upper_fence = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
df["is_outlier"] = ~df["word_count"].between(lower_fence, upper_fence)
print(f"Word count outliers: {df['is_outlier'].sum()} ({df['is_outlier'].mean()*100:.1f}%)")
print(f"IQR fence: [{lower_fence:.0f}, {upper_fence:.0f}]")

Word count outliers: 19994 (2.4%)
IQR fence: [-8, 416]


---
## STEP 4 — Statistical Analysis

> **Goal:** Quantify distributions, central tendency, spread, and group-level summaries.

In [ ]:
# 4.1  Descriptive statistics for numeric features
num_cols = ["word_count", "char_count", "sentence_count",
            "unique_word_count", "avg_word_length", "vocab_richness"]
desc = df[num_cols].describe().round(3)
print("Descriptive Statistics:")
desc

Descriptive Statistics:


,word_count,char_count,sentence_count,unique_word_count,avg_word_length,vocab_richness
count,821116.000,821116.000,821116.000,821116.000,821116.000,821116.000
mean,211.829,1461.517,8.596,86.964,5.960,0.428
std,99.271,684.852,4.884,34.297,2.617,0.119
min,1.000,1.000,1.000,0.000,1.000,0.000
25%,151.000,1051.000,5.000,67.000,5.637,0.376
50%,207.000,1437.000,8.000,86.000,5.906,0.426
75%,257.000,1792.000,11.000,104.000,6.185,0.475
max,18000.000,122392.000,422.000,4246.000,539.000,16.500


In [ ]:
# 4.2  Skewness & Kurtosis
print("\nSkewness & Kurtosis:")
sk = pd.DataFrame({
    "Skewness" : df[num_cols].skew().round(3),
    "Kurtosis" : df[num_cols].kurt().round(3),
})
print(sk)


Skewness & Kurtosis:
                   Skewness   Kurtosis
word_count           15.580   2264.739
char_count           16.714   2523.565
sentence_count        2.022     79.574
unique_word_count     6.548    622.666
avg_word_length      90.781  10723.144
vocab_richness       40.089   3413.059


In [ ]:
# 4.3  Publication year counts
# Convert published_date into datetime
df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")

# Publication counts by year
print("\nPublications per Year:")

year_stats = df["published_date"].dt.year.value_counts().sort_index()

print(year_stats.to_frame("count"))

# Year range
print(f"\nYear range: {df['published_date'].dt.year.min()} – {df['published_date'].dt.year.max()}")


Publications per Year:
                 count
published_date        
1879.0               1
1955.0               1
1957.0               1
1962.0               1
1963.0               1
...                ...
2019.0            5119
2020.0          118437
2021.0          182377
2022.0           74249
2023.0               1

[62 rows x 1 columns]

Year range: 1879.0 – 2023.0


In [ ]:
# 4.4  Topic-level word count statistics
print("\nWord Count by Title (mean | median | std):")
topic_stats = df.groupby("title")["word_count"].agg(["mean","median","std"]).round(1)
print(topic_stats.sort_values("mean", ascending=False).to_string())


Word Count by Title (mean | median | std):


KeyboardInterrupt: 

In [ ]:
# 4.5  Claim label distribution
print("\nClaim Label Distribution:")
label_counts = df["claim_label"].value_counts()
label_pct    = (label_counts / len(df) * 100).round(2)
print(pd.DataFrame({"Count": label_counts, "Percentage %": label_pct}))

In [ ]:
# 4.6  Top 20 most frequent journals
print("\nTop 10 Journals:")
print(df["journal"].value_counts().head(10).to_frame("count"))

---
## STEP 5 — Visualization

> **Goal:** Plot all distributions, class breakdowns, n-grams, word clouds, TF-IDF, and time trends.

In [ ]:
# 5.1  Text Feature Distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Abstract Text Feature Distributions", fontsize=15, fontweight="bold")

feat_info = [
    ("word_count",        "Abstract Word Count",          "Words"),
    ("sentence_count",    "Sentences per Abstract",        "Sentences"),
    ("unique_word_count", "Unique Word Count (cleaned)",   "Unique Words"),
    ("avg_word_length",   "Average Word Length",           "Characters per Word"),
    ("vocab_richness",    "Vocabulary Richness (Unique/Total)", "Ratio"),
    ("char_count",        "Character Count per Abstract",  "Characters"),
]

for ax, (feat, title, xlabel), color in zip(axes.flatten(), feat_info, COLORS):
    ax.hist(df[feat], bins=35, color=color, edgecolor="white", alpha=0.85)
    med = df[feat].median()
    ax.axvline(med, color="red", linestyle="--", label=f"Median = {med:.1f}")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("01_feature_distributions.png", bbox_inches="tight")
plt.show()
print("Saved: 01_feature_distributions.png")

In [ ]:
# 5.2  Topic & Claim Label Distribution
topic_counts = df["topic"].value_counts()

LABEL_COLORS = {
    "Supported"            : "#2ecc71",
    "Refuted"              : "#e74c3c",
    "Not Enough Information": "#f39c12",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Topic & Claim Label Distribution", fontsize=14, fontweight="bold")

# Bar: topic
bars = axes[0].barh(topic_counts.index, topic_counts.values, color=COLORS[:len(topic_counts)])
axes[0].set_title("Abstracts per Topic")
axes[0].set_xlabel("Count")
for bar, v in zip(bars, topic_counts.values):
    axes[0].text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
                 str(v), va="center", fontsize=8)

# Pie: topic
axes[1].pie(topic_counts.values, labels=None, colors=COLORS[:len(topic_counts)],
            autopct="%1.1f%%", startangle=140, pctdistance=0.8)
axes[1].legend(topic_counts.index, loc="lower right", fontsize=6, ncol=2)
axes[1].set_title("Topic Proportion")

# Bar: claim label
axes[2].bar(label_counts.index, label_counts.values,
            color=[LABEL_COLORS[l] for l in label_counts.index], edgecolor="white")
axes[2].set_title("Claim Verification Label")
axes[2].set_ylabel("Count")
for i, v in enumerate(label_counts.values):
    axes[2].text(i, v + 4, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("02_topic_label_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: 02_topic_label_distribution.png")

In [ ]:
# 5.3  Publications Over Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Publications Over Time", fontsize=13, fontweight="bold")

year_counts = df["publish_year"].value_counts().sort_index()
axes[0].bar(year_counts.index.astype(str), year_counts.values, color=COLORS[3], edgecolor="white")
axes[0].set_title("Total Papers per Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")

# Stacked by topic
pivot = df.groupby(["publish_year", "topic"]).size().unstack(fill_value=0)
pivot.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab20", legend=False)
axes[1].set_title("Papers per Year by Topic")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("03_publications_over_time.png", bbox_inches="tight")
plt.show()
print("Saved: 03_publications_over_time.png")

In [ ]:
# 5.4  Top N-gram Analysis (Unigrams, Bigrams, Trigrams)
def get_top_ngrams(corpus, n: int = 1, top_k: int = 15):
    vec = CountVectorizer(ngram_range=(n, n), max_features=5000).fit(corpus)
    bag = vec.transform(corpus)
    sum_words = bag.sum(axis=0)
    freq = [(w, int(sum_words[0, idx])) for w, idx in vec.vocabulary_.items()]
    return sorted(freq, key=lambda x: x[1], reverse=True)[:top_k]

corpus = df["clean_abstract"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle("Top N-grams in CORD-19 Abstracts (after preprocessing)",
             fontsize=14, fontweight="bold")

for ax, (n, label) in zip(axes, [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]):
    ngrams = get_top_ngrams(corpus, n=n, top_k=15)
    words, freqs = zip(*ngrams)
    colors = sns.color_palette("coolwarm", len(words))
    ax.barh(list(reversed(words)), list(reversed(freqs)), color=list(reversed(colors)))
    ax.set_title(f"Top 15 {label}", fontsize=12)
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("04_ngrams.png", bbox_inches="tight")
plt.show()
print("Saved: 04_ngrams.png")

In [ ]:
# 5.5  Word Clouds — All & per Topic
all_text = " ".join(df["clean_abstract"].tolist())

wc = WordCloud(width=1200, height=500, background_color="white",
               colormap="plasma", max_words=150, collocations=False).generate(all_text)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — All COVID-19 Abstracts", fontsize=15, fontweight="bold", pad=12)
plt.savefig("05_wordcloud_all.png", bbox_inches="tight")
plt.show()
print("Saved: 05_wordcloud_all.png")

In [ ]:
# Per-topic word clouds (first 4 topics)
sample_topics = df["topic"].unique()[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Word Clouds by Topic", fontsize=13, fontweight="bold")

for ax, topic in zip(axes.flatten(), sample_topics):
    topic_text = " ".join(df[df["topic"] == topic]["clean_abstract"].tolist())
    wc_t = WordCloud(width=600, height=350, background_color="white",
                     colormap="viridis", max_words=80, collocations=False).generate(topic_text)
    ax.imshow(wc_t, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(topic.title(), fontsize=10)

plt.tight_layout()
plt.savefig("06_wordclouds_per_topic.png", bbox_inches="tight")
plt.show()
print("Saved: 06_wordclouds_per_topic.png")

In [ ]:
# 5.6  TF-IDF Top Terms
tfidf  = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
tfidf_matrix = tfidf.fit_transform(df["clean_abstract"])
feature_names = np.array(tfidf.get_feature_names_out())

mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[-20:][::-1]
top_terms  = feature_names[top_idx]
top_scores = mean_tfidf[top_idx]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_terms[::-1], top_scores[::-1], color=sns.color_palette("rocket", 20))
ax.set_title("Top 20 Terms by Mean TF-IDF Score", fontsize=13, fontweight="bold")
ax.set_xlabel("Mean TF-IDF Score")
for bar, score in zip(bars, top_scores[::-1]):
    ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
            f"{score:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("07_tfidf_top_terms.png", bbox_inches="tight")
plt.show()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape} | Saved: 07_tfidf_top_terms.png")

---
## STEP 6 — Find Patterns & Relationships

> **Goal:** Discover correlations between features, topic clusters, feature-label relationships, and semantic structure via LSA.

In [ ]:
# 6.1  Correlation Heatmap
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Heatmap — Abstract Text Features",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("08_correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("Saved: 08_correlation_heatmap.png")

In [ ]:
# 6.2  Feature Distributions by Verification Label (Boxplots)
order = ["Supported", "Refuted", "Not Enough Information"]
palette = [LABEL_COLORS[l] for l in order]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Abstract Text Features by Verification Label", fontsize=13, fontweight="bold")

for ax, (feat, title) in zip(axes, [
    ("word_count",        "Word Count"),
    ("sentence_count",    "Sentence Count"),
    ("unique_word_count", "Unique Words"),
]):
    sns.boxplot(data=df, x="claim_label", y=feat, order=order,
                palette=palette, ax=ax, width=0.5)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("09_features_by_label.png", bbox_inches="tight")
plt.show()
print("Saved: 09_features_by_label.png")

In [ ]:
# 6.3  Vocabulary Richness by Topic
fig, ax = plt.subplots(figsize=(12, 5))
topic_order = df.groupby("topic")["vocab_richness"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="topic", y="vocab_richness", order=topic_order,
            palette="Set3", width=0.5, ax=ax)
ax.set_title("Vocabulary Richness Distribution by Topic", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("10_vocab_richness_by_topic.png", bbox_inches="tight")
plt.show()
print(" Saved: 10_vocab_richness_by_topic.png")

In [ ]:
# 6.4  Pairplot of numeric features (sample)
pairplot_df = df[num_cols + ["claim_label"]].sample(500, random_state=42)
g = sns.pairplot(pairplot_df, hue="claim_label",
                 palette=LABEL_COLORS, plot_kws={"alpha": 0.4, "s": 15},
                 diag_kind="kde", vars=["word_count","unique_word_count","vocab_richness"])
g.fig.suptitle("Pairplot of Key Text Features by Claim Label", y=1.02, fontsize=12)
g.fig.savefig("11_pairplot.png", bbox_inches="tight")
plt.show()
print("Saved: 11_pairplot.png")

In [ ]:
# 6.5  Latent Semantic Analysis (SVD) — Thematic Structure
n_components = 8
svd = TruncatedSVD(n_components=n_components, random_state=42)
lsa_matrix = svd.fit_transform(tfidf_matrix)

print(f"Explained variance (first {n_components} components): "
      f"{svd.explained_variance_ratio_.sum():.3f}")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("LSA — Top Terms per Latent Component (TF-IDF + SVD)",
             fontsize=13, fontweight="bold")

for i, ax in enumerate(axes.flatten()):
    comp = svd.components_[i]
    top_idx_c = comp.argsort()[-10:][::-1]
    top_names = feature_names[top_idx_c]
    top_vals  = comp[top_idx_c]
    ax.barh(top_names[::-1], top_vals[::-1], color=COLORS[i % len(COLORS)])
    ax.set_title(f"Component {i+1}", fontsize=10)
    ax.set_xlabel("Weight")

plt.tight_layout()
plt.savefig("12_lsa_components.png", bbox_inches="tight")
plt.show()
print("Saved: 12_lsa_components.png")

In [ ]:
# 6.6  LSA 2D Scatter — Semantic Clusters
topic_ids, topic_labels = pd.factorize(df["topic"])

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(lsa_matrix[:, 0], lsa_matrix[:, 1],
                     c=topic_ids, cmap="tab10", alpha=0.50, s=14)
ax.set_title("LSA 2D Projection — Abstracts Coloured by Topic",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Component 1")
ax.set_ylabel("Component 2")

handles = [plt.Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=plt.cm.tab10(i / len(topic_labels)),
                      markersize=8, label=t)
           for i, t in enumerate(topic_labels)]
ax.legend(handles=handles, fontsize=7, loc="upper right", ncol=2)
plt.tight_layout()
plt.savefig("13_lsa_2d_scatter.png", bbox_inches="tight")
plt.show()
print("Saved: 13_lsa_2d_scatter.png")

In [ ]:
# 6.7  Cosine Similarity — Evidence Retrieval Simulation
SAMPLE_CLAIM = "COVID-19 vaccines significantly reduce severe illness and hospitalization."

claim_clean = preprocess_text(SAMPLE_CLAIM)
claim_vec   = tfidf.transform([claim_clean])
sims        = cosine_similarity(claim_vec, tfidf_matrix).flatten()
df["cosine_sim"] = sims

TOP_K = 10
top_idx = sims.argsort()[-TOP_K:][::-1]
top_df  = df.iloc[top_idx][["topic", "cosine_sim", "claim_label"]].copy()
top_df["rank"] = [f"#{i+1}" for i in range(TOP_K)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Cosine Similarity — Claim: "{SAMPLE_CLAIM}"',
             fontsize=10, fontweight="bold")

axes[0].hist(sims, bins=50, color=COLORS[0], edgecolor="white")
axes[0].axvline(0.20, color="red", linestyle="--", label="Retrieval threshold 0.20")
axes[0].set_title("Similarity Distribution across All Abstracts")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].barh(top_df["rank"] + "  " + top_df["topic"],
             top_df["cosine_sim"], color=COLORS[1])
axes[1].set_title(f"Top-{TOP_K} Retrieved Abstracts")
axes[1].set_xlabel("Cosine Similarity")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("14_cosine_similarity.png", bbox_inches="tight")
plt.show()

print(f"\nTop-{TOP_K} retrieved abstracts:")
print(top_df[["rank","topic","cosine_sim","claim_label"]].to_string(index=False))
print("Saved: 14_cosine_similarity.png")

---
## STEP 7 — Prepare for Modeling

> **Goal:** Encode labels, scale numeric features, and build the final feature matrix ready to feed into a classifier.

In [ ]:
# 7.1  Encode the target label
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["claim_label"])

print("Label Encoding:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")
print(f"\nLabel distribution (encoded):")
print(df["label_encoded"].value_counts().sort_index())

In [ ]:
# 7.2  Scale numeric features with MinMaxScaler
scaler = MinMaxScaler()
df[[f"{c}_scaled" for c in num_cols]] = scaler.fit_transform(df[num_cols])

print("Scaled numeric features (first 3 rows):")
df[[f"{c}_scaled" for c in num_cols]].head(3).round(4)

In [ ]:
# 7.3  Build final feature matrix for modeling
#       Combine: TF-IDF sparse matrix  +  scaled numeric features  +  LSA components

import scipy.sparse as sp

scaled_cols = [f"{c}_scaled" for c in num_cols]
numeric_sparse = sp.csr_matrix(df[scaled_cols].values)
lsa_sparse     = sp.csr_matrix(lsa_matrix)

X = sp.hstack([tfidf_matrix, numeric_sparse, lsa_sparse])
y = df["label_encoded"].values

print(f"Feature matrix X shape : {X.shape}")
print(f"   → TF-IDF features      : {tfidf_matrix.shape[1]}")
print(f"   → Scaled numeric feats : {len(scaled_cols)}")
print(f"   → LSA components       : {n_components}")
print(f"   Target y shape         : {y.shape}")
print(f"   Classes                : {list(le.classes_)}")

In [ ]:
# 7.4  Train / Validation / Test Split
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Dataset Splits:")
print(f"  Train : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(y)*100:.0f}%)")
print(f"  Val   : {X_val.shape[0]:,}   rows ({X_val.shape[0]/len(y)*100:.0f}%)")
print(f"  Test  : {X_test.shape[0]:,}   rows ({X_test.shape[0]/len(y)*100:.0f}%)")

In [ ]:
# 7.5  Class balance check in splits
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Label Distribution Across Train / Val / Test Splits",
             fontsize=12, fontweight="bold")

for ax, (split_y, split_name) in zip(axes,
    [(y_train, "Train"), (y_val, "Validation"), (y_test, "Test")]):
    counts = np.bincount(split_y)
    ax.bar(le.classes_, counts,
           color=[LABEL_COLORS[c] for c in le.classes_], edgecolor="white")
    ax.set_title(f"{split_name} ({len(split_y):,} samples)")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)
    for i, v in enumerate(counts):
        ax.text(i, v + 1, str(v), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("15_split_label_balance.png", bbox_inches="tight")
plt.show()
print("Saved: 15_split_label_balance.png")

In [ ]:
# 7.6  Save the preprocessed dataframe & feature matrix
import pickle

df.to_csv("cord19_preprocessed.csv", index=False)

with open("modeling_artifacts.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "tfidf": tfidf, "svd": svd, "scaler": scaler,
        "label_encoder": le,
    }, f)

print("Saved:")
print("   cord19_preprocessed.csv  — cleaned & feature-engineered dataframe")
print("   modeling_artifacts.pkl   — train/val/test splits + fitted transformers")

---
## EDA Summary Report

In [ ]:
summary = {
    "Total Abstracts"                  : len(df),
    "Publication Year Range"           : f"{df['publish_year'].min()}–{df['publish_year'].max()}",
    "Unique Topics"                    : df['topic'].nunique(),
    "Unique Journals"                  : df['journal'].nunique(),
    "Average Word Count per Abstract"  : f"{df['word_count'].mean():.1f}",
    "Median Word Count"                : f"{df['word_count'].median():.0f}",
    "Average Sentence Count"           : f"{df['sentence_count'].mean():.1f}",
    "Mean Vocabulary Richness"         : f"{df['vocab_richness'].mean():.3f}",
    "Outlier Abstracts (word count)"   : int(df['is_outlier'].sum()),
    "TF-IDF Vocabulary Size"           : tfidf_matrix.shape[1],
    "LSA Explained Variance (8 comp)"  : f"{svd.explained_variance_ratio_.sum():.3f}",
    "Supported Claims"                 : int(label_counts.get('Supported', 0)),
    "Refuted Claims"                   : int(label_counts.get('Refuted', 0)),
    "NEI Claims"                       : int(label_counts.get('Not Enough Information', 0)),
    "Final Feature Matrix Shape"       : f"{X.shape}",
    "Train / Val / Test Sizes"         : f"{X_train.shape[0]} / {X_val.shape[0]} / {X_test.shape[0]}",
}

print("=" * 62)
print("     COVID-19 ClaimCheck EDA — Final Summary Report")
print("=" * 62)
for k, v in summary.items():
    print(f"  {k:<42}: {v}")
print("=" * 62)
print("\n✅ All figures saved to working directory.")
print("\n📌 Next Steps:")
print("   1. Replace demo data with real CORD-19 metadata.csv")
print("   2. Generate SciBERT / SBERT embeddings (Step 2 of algorithm)")
print("   3. Build FAISS index for fast approximate nearest-neighbour search")
print("   4. Fine-tune NLI model (MedNLI / SciFact) for claim classification")
print("   5. Build Flask / Streamlit web interface for deployment")